## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        initializer = tf.initializers.TruncatedNormal(stddev=0.1)

        self.W1 = tf.Variable(initializer(shape=[784, 100]))
        self.b1 = tf.Variable(tf.zeros([100]))

        self.W2 = tf.Variable(initializer(shape=[100, 10]))
        self.b2 = tf.Variable(tf.zeros([10]))

    def __call__(self, x):
        x = tf.reshape(x, [-1, 784])
        
        h1 = tf.matmul(x, self.W1) + self.b1
        h1_relu = tf.nn.relu(h1)
        
        logits = tf.matmul(h1_relu, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.3787842 ; accuracy 0.13543333
epoch 1 : loss 2.3692405 ; accuracy 0.13883333
epoch 2 : loss 2.3600338 ; accuracy 0.14233333
epoch 3 : loss 2.3511324 ; accuracy 0.14566667
epoch 4 : loss 2.342508 ; accuracy 0.14948334
epoch 5 : loss 2.3341374 ; accuracy 0.15331666
epoch 6 : loss 2.325999 ; accuracy 0.15783334
epoch 7 : loss 2.318073 ; accuracy 0.16125
epoch 8 : loss 2.3103433 ; accuracy 0.16561666
epoch 9 : loss 2.3027935 ; accuracy 0.17015
epoch 10 : loss 2.2954097 ; accuracy 0.1745
epoch 11 : loss 2.2881782 ; accuracy 0.1794
epoch 12 : loss 2.2810872 ; accuracy 0.18428333
epoch 13 : loss 2.274126 ; accuracy 0.18906666
epoch 14 : loss 2.267285 ; accuracy 0.19403334
epoch 15 : loss 2.2605567 ; accuracy 0.1989
epoch 16 : loss 2.2539322 ; accuracy 0.20425
epoch 17 : loss 2.247406 ; accuracy 0.20906667
epoch 18 : loss 2.2409708 ; accuracy 0.21411666
epoch 19 : loss 2.2346215 ; accuracy 0.2191
epoch 20 : loss 2.2283516 ; accuracy 0.2241
epoch 21 : loss 2.2221577 ; accuracy 